# Eye Tracking benchmark: 1. Construct the reference outcome table

This notebook constructs the versioned artificial-gap benchmark for the Eye Tracking domain. It implements the frozen reference protocol, writes derived outputs only, and never copies external source data into the repository.

**Purpose.** The resulting table combines observable local context, per-method outcomes, and gap provenance. It is the shared input to the descriptive analysis, confirmatory nested evaluation, and final deployment fit in notebooks 2–4.


## Data scope and eligibility

The frozen reference covers GazeBase, GazeBaseVR, ZuCo 2.0, and Pedrotti et al. Source files remain external and are loaded under the documented raw-data patterns. Native coordinate systems are retained deliberately, so the cross-dataset setting is heterogeneous by design.

The protocol selects 10 participants per dataset. GazeBase, GazeBaseVR, and ZuCo contribute two recordings per participant with 20 gaps per recording; Pedrotti contributes 40 designated trials per participant. It requests 400 gaps per dataset across five duration strata (0–250 ms, where zero means one sample).


## Reproducible inputs and deliberate rebuilds

`BENCHMARK_DATA_DIR` in `.env` must point to the required external data root. `EYE_TRACKING_BENCHMARK_DIR` can optionally redirect the derived benchmark output; the published default is `benchmarks/eyetracking`.

The frozen protocol is defined in `configs/eye_tracking_final.toml`. It fixes sampling, context requirements, duration strata, random seed, and the scale-floor quantile. The builder reads the ignored local raw-data root and uses the frozen participant selection, recording order, seed, and 100-attempt sampling limit. No source file is altered.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

# Read ignored workstation paths without placing them in this notebook.
source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
from gap_imputation_benchmark.paths import load_local_environment, local_path_or_default
# The local .env file is authoritative for this workstation notebook.
load_local_environment(override=True)

required_paths = {'BENCHMARK_DATA_DIR': 'a local directory containing raw/'}
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    requirements = '\n'.join(f'- {name}: {required_paths[name]}' for name in missing_paths)
    raise RuntimeError(
        'Set the following environment variable(s) in .env or in the current session:\n'
        f'{requirements}'
    )

BENCHMARK_DIR = local_path_or_default(
    'EYE_TRACKING_BENCHMARK_DIR', PROJECT_ROOT / 'benchmarks' / 'eyetracking',
)
CONFIG = PROJECT_ROOT / 'configs' / 'eye_tracking_final.toml'
SCRIPT = PROJECT_ROOT / 'scripts' / 'build_eye_tracking_benchmark.py'

# Run directly from src/ so the notebook works before an editable package install.
execution_env = os.environ.copy()
execution_env['PYTHONPATH'] = source_root + os.pathsep + execution_env.get('PYTHONPATH', '')
command = [sys.executable, str(SCRIPT), '--config', str(CONFIG), '--output-dir', str(BENCHMARK_DIR)]
subprocess.run(command, cwd=PROJECT_ROOT, env=execution_env, check=True)
try:
    output_location = BENCHMARK_DIR.relative_to(PROJECT_ROOT).as_posix()
except ValueError:
    output_location = 'configured benchmark directory'
print(f'Benchmark outputs: {output_location}')

[load] GazeBase: 644 source files
[load] GazeBase: 1/644
[load] GazeBase: 32/644
[load] GazeBase: 64/644
[load] GazeBase: 96/644
[load] GazeBase: 128/644
[load] GazeBase: 160/644
[load] GazeBase: 192/644
[load] GazeBase: 224/644
[load] GazeBase: 256/644
[load] GazeBase: 288/644
[load] GazeBase: 320/644
[load] GazeBase: 352/644
[load] GazeBase: 384/644
[load] GazeBase: 416/644
[load] GazeBase: 448/644
[load] GazeBase: 480/644
[load] GazeBase: 512/644
[load] GazeBase: 544/644
[load] GazeBase: 576/644
[load] GazeBase: 608/644
[load] GazeBase: 640/644
[load] GazeBase: 644/644
[load] GazeBaseVR: 1004 source files
[load] GazeBaseVR: 1/1004
[load] GazeBaseVR: 50/1004
[load] GazeBaseVR: 100/1004
[load] GazeBaseVR: 150/1004
[load] GazeBaseVR: 200/1004
[load] GazeBaseVR: 250/1004
[load] GazeBaseVR: 300/1004
[load] GazeBaseVR: 350/1004
[load] GazeBaseVR: 400/1004
[load] GazeBaseVR: 450/1004
[load] GazeBaseVR: 500/1004
[load] GazeBaseVR: 550/1004
[load] GazeBaseVR: 600/1004
[load] GazeBaseVR: 650/

## Generate the benchmark

The canonical script evaluates the same artificial-gap geometry for every registered candidate method: forward fill, nearest boundary, linear interpolation, PCHIP, local natural cubic spline, BIC-selected polynomial, and template reconstruction. No seasonal reference is registered for the Eye-Tracking domain. Each gap retains portable source references and diagnostics needed for review.


In [2]:
import json
import pandas as pd

display(pd.read_csv(BENCHMARK_DIR / 'coverage_table.csv'))
display(pd.read_csv(BENCHMARK_DIR / 'dataset_summary.csv'))
metadata = json.loads((BENCHMARK_DIR / 'metadata.json').read_text(encoding='utf-8'))
metadata

,dataset_id,requested_gaps,learnable_gaps,excluded_gaps
0,GazeBase,400,400,0
1,GazeBaseVR,400,400,0
2,Pedrotti,400,399,1
3,ZuCo,400,400,0


,dataset_id,gaps,participants,recordings
0,GazeBase,400,10,20
1,GazeBaseVR,400,10,20
2,Pedrotti,399,10,399
3,ZuCo,400,10,20


{'artifact_type': 'benchmark',
 'domain': 'eye_tracking',
 'workflow': 'eye_tracking_benchmark',
 'config': {'min_gap_duration_ms': 0.0,
  'max_gap_duration_ms': 250.0,
  'n_gaps_per_recording': 20,
  'n_strata': 5,
  'min_context_valid_fraction': 0.8,
  'random_state': 42,
  'participants_per_dataset': 10,
  'recordings_per_participant': 2,
  'pedrotti_recordings_per_participant': 40,
  'pedrotti_gaps_per_participant': 40,
  'max_gap_sampling_attempts': 100},
 'random_state': 42,
 'method_names': ['forward_fill',
  'nearest_boundary',
  'linear',
  'pchip',
  'local_natural_cubic_spline',
  'polyfit_bic',
  'template'],
 'feature_columns': ['realized_gap_duration_ms',
  'left_context_valid_fraction',
  'right_context_valid_fraction',
  'normalized_boundary_jump',
  'normalized_mean_difference_right_minus_left',
  'local_std_over_scale',
  'local_range_over_scale',
  'normalized_trend_before',
  'normalized_trend_after',
  'normalized_trend_difference',
  'trend_before_r2',
  'trend_af

## Review the generated reference tables

`coverage_table.csv` verifies requested, learnable, and excluded gaps at the relevant domain level. `selected_recordings.csv` records the deterministically selected source units; `input_manifest.csv` records input discovery and portable references; `metadata.json` records the data scope and frozen protocol.

Any sampling shortfall or post-sampling exclusion remains explicit in the output files. Review these records before treating a rerun as equivalent to the published reference.
